# Une grille de planning sans IA : OpenCV contre YOLO

**Question du jour : faut-il toujours de l'IA ?** Un planning imprimé avec des bordures est... une grille.
Des traits horizontaux, des traits verticaux, des cases. Avant d'entraîner un modèle, voyons ce que l'on obtient
avec du **traitement d'image classique** (OpenCV) : aucun poids appris, aucune annotation, aucun GPU.

```
photo → niveaux de gris → binarisation → traits H / V → intersections → redressement → cases → rôles
```

À la fin du notebook, vous aurez :

1. suivi chaque étape en la visualisant ;
2. obtenu des boîtes dans **le même format que la sortie de votre modèle YOLO** (classes `event`, `date_cell`,
   `resource_header`) : elles peuvent alimenter l'OCR puis le parsing du jour 3 ;
3. mesuré le **rappel** (cases retrouvées / cases attendues), la métrique qui compte pour nous : une case ratée est un créneau perdu ;
4. identifié les cas où cette approche **échoue** et où l'IA reprend l'avantage.

Prérequis : `pip install opencv-python-headless numpy matplotlib`.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Votre propre photo de planning à codes (une ligne par personne, une colonne par jour, avec des bordures).
# None = planning de démonstration dessiné plus bas.
IMAGE = None

## 1. Un planning de démonstration (et sa vérité terrain)

Comme nous dessinons le planning nous-mêmes, nous connaissons **exactement** la position de chaque case non vide :
c'est ce qui permettra de mesurer le rappel. `simuler_photo` fait « mal prendre » la photo : feuille posée sur une table,
vue en perspective, un peu penchée, ombrée, bruitée.

In [ ]:
JOURS = ["L04", "M05", "M06", "J07", "V08", "S09", "D10", "L11", "M12", "M13", "J14", "V15"]
PERSONNES = ["Babin", "Jouan", "Henneuse", "Bediee", "Gayet", "Laronce", "Novak"]
CODES = ["M06", "J09", "N08", "RH", "G03", "CA", "S14", "MAL"]
X0, Y0, LARG_NOMS, LARG_JOUR, H_ENTETE, H_LIGNE = 40, 90, 150, 80, 50, 64


def dessiner_planning(bordures=True, seed=0):
    # Retourne (image BGR, centres (x, y) des cases non vides).
    rng = np.random.default_rng(seed)
    nl, nc = len(PERSONNES), len(JOURS)
    img = np.full((Y0 + H_ENTETE + nl * H_LIGNE + 60, X0 + LARG_NOMS + nc * LARG_JOUR + 40, 3), 255, np.uint8)
    font = cv2.FONT_HERSHEY_SIMPLEX
    cv2.putText(img, "Planning EHPAD - Janvier 2027", (X0, 55), font, 0.9, (0, 0, 0), 2)
    xs = [X0, X0 + LARG_NOMS] + [X0 + LARG_NOMS + (j + 1) * LARG_JOUR for j in range(nc)]
    ys = [Y0, Y0 + H_ENTETE] + [Y0 + H_ENTETE + (i + 1) * H_LIGNE for i in range(nl)]
    if bordures:
        for x in xs:
            cv2.line(img, (x, ys[0]), (x, ys[-1]), (0, 0, 0), 1)
        for y in ys:
            cv2.line(img, (xs[0], y), (xs[-1], y), (0, 0, 0), 1)
    for j, jour in enumerate(JOURS):
        cv2.putText(img, jour, (xs[j + 1] + 15, ys[0] + 32), font, 0.6, (0, 0, 0), 2)
    centres = []
    for i, nom in enumerate(PERSONNES):
        cy = ys[i + 1] + H_LIGNE // 2
        cv2.putText(img, nom, (xs[0] + 8, cy + 8), font, 0.6, (0, 0, 0), 2)
        for j in range(nc):
            if rng.random() < 0.12:          # case vide (repos non noté, absence...)
                continue
            cv2.putText(img, str(rng.choice(CODES)), (xs[j + 1] + 15, cy + 8), font, 0.6, (0, 0, 0), 2)
            centres.append((xs[j + 1] + LARG_JOUR / 2, cy))
    return img, centres


def simuler_photo(img, centres, angle=2.5, seed=0):
    # Feuille sur une table, perspective + rotation, ombre, flou, bruit. Retourne (photo, centres transformés).
    rng = np.random.default_rng(seed)
    h, w = img.shape[:2]
    pad = 80
    toile = np.full((h + 2 * pad, w + 2 * pad, 3), (70, 95, 120), np.uint8)      # la table
    toile[pad:pad + h, pad:pad + w] = img
    ch, cw = toile.shape[:2]
    src = np.float32([[0, 0], [cw, 0], [cw, ch], [0, ch]])
    H = cv2.getPerspectiveTransform(src, src + (rng.uniform(-0.03, 0.03, (4, 2)) * [cw, ch]).astype(np.float32))
    R = np.vstack([cv2.getRotationMatrix2D((cw / 2, ch / 2), angle, 1.0), [0, 0, 1]])
    H = R @ H
    photo = cv2.warpPerspective(toile, H, (cw, ch), borderMode=cv2.BORDER_REPLICATE).astype(np.float32)
    gx, gy = np.meshgrid(np.linspace(0, 1, cw), np.linspace(0, 1, ch))
    photo *= (1.08 - 0.25 * (0.6 * gx + 0.4 * gy))[..., None]                     # ombre douce
    photo = cv2.GaussianBlur(photo, (0, 0), 1.0) + rng.normal(0, 4, photo.shape)  # flou + bruit capteur
    pts = cv2.perspectiveTransform(np.float32([[x + pad, y + pad] for x, y in centres]).reshape(-1, 1, 2), H)
    return np.clip(photo, 0, 255).astype(np.uint8), [tuple(p) for p in pts.reshape(-1, 2)]


def voir(images, titres=None, taille=5):
    fig, axes = plt.subplots(1, len(images), figsize=(taille * len(images), taille * 0.75), squeeze=False)
    for k, (ax, im) in enumerate(zip(axes[0], images)):
        ax.imshow(im if im.ndim == 2 else cv2.cvtColor(im, cv2.COLOR_BGR2RGB), cmap="gray")
        ax.set_title(titres[k] if titres else "")
        ax.axis("off")
    plt.tight_layout()
    plt.show()


if IMAGE:
    photo, verite = cv2.imread(IMAGE), None
else:
    propre, centres = dessiner_planning()
    photo, verite = simuler_photo(propre, centres)
    voir([propre, photo], ["Impression à plat", "« Photo » (perspective, ombre, bruit)"], taille=7)
    print(f"{len(verite)} cases non vides attendues")

## 2. Étape 1 : de la photo à une image binaire

On passe en niveaux de gris, puis on garde seulement « ce qui est de l'encre » : un pixel est de l'encre s'il est nettement
plus sombre que ses voisins. C'est le **seuillage adaptatif** : le seuil suit l'éclairage local, donc une ombre sur
un coin de la photo ne le gêne pas (un seuil global, lui, perdrait tout un côté).

In [ ]:
def binariser(img):
    gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gris = cv2.medianBlur(gris, 3)                                   # enlève le bruit isolé
    bloc = max(15, (min(gris.shape) // 40) | 1)                      # taille du « voisinage » (impaire)
    return cv2.adaptiveThreshold(gris, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY_INV, bloc, 10)


seuil_global = cv2.threshold(cv2.cvtColor(photo, cv2.COLOR_BGR2GRAY), 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
binaire = binariser(photo)
voir([seuil_global, binaire], ["Seuil global (Otsu) : l'ombre gêne", "Seuil adaptatif : encre = blanc"], taille=7)

## 3. Étape 2 : isoler les traits par morphologie

Une **ouverture morphologique** avec un noyau très long et très fin (par exemple 1 pixel de haut et 1/15 de la largeur
de l'image) ne laisse survivre que les segments **horizontaux plus longs que le noyau**. Le texte (des petits traits
courts) disparaît, les lignes de la grille restent. Même chose avec un noyau vertical pour les colonnes.

Astuce pour les photos : un trait penché de 2 ou 3 degrés « dérive » de plusieurs pixels sur la longueur du noyau et
serait coupé. On **épaissit** donc d'abord un peu l'image dans le sens perpendiculaire (`inclinaison`).

In [ ]:
def extraire_lignes(binaire, longueur, inclinaison=0.08):
    # Retourne (traits horizontaux, traits verticaux) sous forme de masques 0 / 255.
    ep = max(1, int(longueur * inclinaison) | 1)
    noyau_h = cv2.getStructuringElement(cv2.MORPH_RECT, (longueur, 1))
    noyau_v = cv2.getStructuringElement(cv2.MORPH_RECT, (1, longueur))
    horiz = cv2.morphologyEx(cv2.dilate(binaire, np.ones((ep, 1), np.uint8)), cv2.MORPH_OPEN, noyau_h)
    vert = cv2.morphologyEx(cv2.dilate(binaire, np.ones((1, ep), np.uint8)), cv2.MORPH_OPEN, noyau_v)
    for m in (horiz, vert):                      # le bord de l'image n'est pas un trait
        m[:3, :] = m[-3:, :] = 0
        m[:, :3] = m[:, -3:] = 0
    return horiz, vert


longueur = min(photo.shape[:2]) // 15
horiz, vert = extraire_lignes(binaire, longueur)
voir([horiz, vert], [f"Traits horizontaux (noyau {longueur} x 1)", f"Traits verticaux (noyau 1 x {longueur})"], taille=7)

## 4. Étape 3 : trouver le tableau, puis le redresser

Là où un trait horizontal **croise** un trait vertical, il y a un coin de case : on obtient des points
(`cv2.bitwise_and`). Attention : le **bord de la feuille** est aussi un long trait, mais il n'a que 4 intersections.
Le tableau, lui, en a des dizaines (lignes × colonnes) : on retient l'**amas d'intersections le plus peuplé**.

Ses 4 coins extrêmes (haut-gauche = plus petit `x + y`, bas-droit = plus grand `x + y`, etc.) donnent un quadrilatère. 
`cv2.getPerspectiveTransform` calcule la **homographie** qui le transforme en rectangle parfait ; 
`cv2.warpPerspective` l'applique : la photo penchée devient un tableau « vu de face ».

In [ ]:
def trouver_tableau(horiz, vert):
    # Retourne les 4 coins du tableau (haut-gauche, haut-droit, bas-droit, bas-gauche) ou None.
    k = np.ones((5, 5), np.uint8)
    croisements = cv2.bitwise_and(cv2.dilate(horiz, k), cv2.dilate(vert, k))
    _, _, _, centres = cv2.connectedComponentsWithStats(croisements)
    centres = centres[1:]
    if len(centres) < 6:
        return None
    # Amas : traits reliés entre eux (la feuille et le tableau sont séparés par la marge)
    amas = cv2.dilate(cv2.bitwise_or(horiz, vert), np.ones((3, 3), np.uint8), iterations=2)
    _, etiquettes = cv2.connectedComponents(amas)
    proprio = np.array([etiquettes[int(y), int(x)] for x, y in centres])
    compte = np.bincount(proprio, minlength=etiquettes.max() + 1)
    compte[0] = 0
    pts = centres[proprio == np.argmax(compte)].astype(np.float32)
    if len(pts) < 6:
        return None
    s, d = pts.sum(axis=1), pts[:, 0] - pts[:, 1]
    return np.float32([pts[s.argmin()], pts[d.argmax()], pts[s.argmax()], pts[d.argmin()]])


MARGE = 20     # marge autour du tableau redressé (les bords de l'image sont ignorés plus haut)


def redresser(img, coins):
    tl, tr, br, bl = coins
    larg = int(max(np.linalg.norm(tr - tl), np.linalg.norm(br - bl)))
    haut = int(max(np.linalg.norm(bl - tl), np.linalg.norm(br - tr)))
    dst = np.float32([[MARGE, MARGE], [MARGE + larg, MARGE], [MARGE + larg, MARGE + haut], [MARGE, MARGE + haut]])
    H = cv2.getPerspectiveTransform(coins, dst)
    return cv2.warpPerspective(img, H, (larg + 2 * MARGE, haut + 2 * MARGE), borderMode=cv2.BORDER_REPLICATE), H


coins = trouver_tableau(horiz, vert)
print("Coins du tableau :", coins.round().tolist())
vue = photo.copy()
cv2.polylines(vue, [coins.astype(np.int32)], True, (0, 0, 255), 3)
redresse, H = redresser(photo, coins)
voir([vue, redresse], ["Tableau trouvé (rouge)", "Tableau redressé"], taille=7)

## 5. Étape 4 : les lignes de la grille

Sur l'image redressée, les traits sont droits. Il suffit de les **projeter** : pour chaque rangée de pixels, quelle part
de la largeur est traversée par un trait horizontal ? Une ligne de la grille donne un pic proche de 100 %, le texte reste
près de 0. Chaque pic = une ligne ; les lignes consécutives délimitent les cases.

In [ ]:
def pics(profil, seuil=0.5):
    # Positions (milieu de chaque plage au-dessus du seuil) d'un profil 1D.
    actif = np.append(profil > seuil, False)
    positions, debut = [], None
    for i, a in enumerate(actif):
        if a and debut is None:
            debut = i
        elif not a and debut is not None:
            positions.append((debut + i - 1) / 2)
            debut = None
    return positions


def lignes_grille(redresse):
    binaire_r = binariser(redresse)
    h, v = extraire_lignes(binaire_r, min(redresse.shape[:2]) // 15, inclinaison=0.04)
    # on ne regarde que l'intérieur du tableau (sans la marge)
    rangees = pics((h[:, MARGE:-MARGE] > 0).mean(axis=1))
    colonnes = pics((v[MARGE:-MARGE, :] > 0).mean(axis=0))
    return rangees, colonnes, binaire_r


rangees, colonnes, binaire_r = lignes_grille(redresse)
print(len(rangees) - 1, "lignes de cases (1 en-tête + personnes) ;", len(colonnes) - 1, "colonnes (1 noms + jours)")
trace = redresse.copy()
for y in rangees:
    cv2.line(trace, (0, int(y)), (trace.shape[1], int(y)), (0, 0, 255), 1)
for x in colonnes:
    cv2.line(trace, (int(x), 0), (int(x), trace.shape[0]), (255, 0, 0), 1)
voir([trace], ["Lignes (rouge) et colonnes (bleu) retrouvées"], taille=9)

## 6. Étape 5 : le rôle de chaque case, par la **structure**

Nous n'avons pas appris à reconnaître une case « événement » : nous le déduisons de sa place dans la grille.

| Place dans la grille | Rôle (même nom de classe que le modèle de référence) |
|---|---|
| 1re ligne, colonnes de jours | `date_cell` (en-tête de jour) |
| 1re colonne, lignes suivantes | `resource_header` (la personne) |
| autres cases **contenant de l'encre** | `event` (le code du créneau) |
| autres cases vides | pas de détection : rien à lire |

« Contient de l'encre » = au moins 1,2 % de pixels blancs dans la binarisation, au centre de la case (les traits de la grille
sont exclus par une marge). Enfin on **remet les boîtes dans l'image d'origine** avec l'homographie inverse, au format de la sortie
OCR du jour 2 (`classe`, `confiance_detection`, `x1`...`y2`).

In [ ]:
def encre(binaire_r, x1, y1, x2, y2):
    mx, my = max(2, 0.15 * (x2 - x1)), max(2, 0.15 * (y2 - y1))
    zone = binaire_r[int(y1 + my):int(y2 - my), int(x1 + mx):int(x2 - mx)]
    return float((zone > 0).mean()) if zone.size else 0.0


def classer_cases(binaire_r, rangees, colonnes, H, seuil_encre=0.012):
    H_inv = np.linalg.inv(H)
    sortie = []
    for i, (y1, y2) in enumerate(zip(rangees[:-1], rangees[1:])):
        for j, (x1, x2) in enumerate(zip(colonnes[:-1], colonnes[1:])):
            if i == 0 and j == 0:
                continue                                         # coin « NOM / PRENOM »
            if i == 0:
                classe = "date_cell"
            elif j == 0:
                classe = "resource_header"
            else:
                classe = "event"
            if encre(binaire_r, x1, y1, x2, y2) < seuil_encre:
                continue                                         # case vide
            coins_case = np.float32([[x1, y1], [x2, y1], [x2, y2], [x1, y2]]).reshape(-1, 1, 2)
            p = cv2.perspectiveTransform(coins_case, H_inv).reshape(-1, 2)   # retour dans la photo
            sortie.append({"classe": classe, "confiance_detection": 0.9,
                           "x1": float(p[:, 0].min()), "y1": float(p[:, 1].min()),
                           "x2": float(p[:, 0].max()), "y2": float(p[:, 1].max())})
    return sortie


zones = classer_cases(binaire_r, rangees, colonnes, H)
from collections import Counter
print(dict(Counter(z["classe"] for z in zones)))
print(zones[0])

In [ ]:
COULEURS = {"event": (40, 120, 250), "date_cell": (60, 200, 60), "resource_header": (200, 160, 0)}


def dessiner(img, zones):
    out = img.copy()
    for z in zones:
        cv2.rectangle(out, (int(z["x1"]), int(z["y1"])), (int(z["x2"]), int(z["y2"])), COULEURS[z["classe"]], 2)
    return out


voir([dessiner(photo, zones)], ["Boîtes retrouvées sans IA (orange = event, vert = date_cell, bleu = personne)"], taille=10)

## 7. Tout assembler et mesurer le rappel

On regroupe les étapes dans une seule fonction, puis on compte : **rappel** = cases attendues qui tombent dans une boîte `event`
détectée ; **précision** = boîtes `event` détectées qui contiennent bien un code. (Ici, le critère est simple : le centre de la
case attendue est dans la boîte, une boîte ne sert qu'une fois.)

In [ ]:
def detecter_grille(img):
    # Photo -> liste de zones (même format que la sortie OCR du jour 2, sans le texte). [] si pas de grille.
    binaire = binariser(img)
    horiz, vert = extraire_lignes(binaire, min(img.shape[:2]) // 15)
    coins = trouver_tableau(horiz, vert)
    if coins is None:
        return []
    redresse, H = redresser(img, coins)
    rangees, colonnes, binaire_r = lignes_grille(redresse)
    if len(rangees) < 3 or len(colonnes) < 3:
        return []
    return classer_cases(binaire_r, rangees, colonnes, H)


def rappel_precision(zones, verite):
    boites = [z for z in zones if z["classe"] == "event"]
    utilisees, trouvees = set(), 0
    for x, y in verite:
        for k, z in enumerate(boites):
            if k not in utilisees and z["x1"] <= x <= z["x2"] and z["y1"] <= y <= z["y2"]:
                utilisees.add(k)
                trouvees += 1
                break
    return trouvees / len(verite), (len(utilisees) / len(boites) if boites else 0.0)


r, p = rappel_precision(detecter_grille(photo), verite)
print(f"Rappel {r:.1%}  Précision {p:.1%}")

### Et sur 20 photos différentes ?

Un seul exemple ne prouve rien (et ce planning dessiné par nos soins est un cas facile : 100 % ici ne promet pas 100 % sur vos photos). On génère 20 plannings (codes différents) et 20 « façons de prendre la photo »
(angle, perspective, ombre, bruit) et on cumule : c'est exactement ce qu'on ferait avec un jeu de test annoté.

In [ ]:
total_att = total_ok = total_det = total_juste = 0
echecs = 0
for graine in range(20):
    propre, centres = dessiner_planning(seed=graine)
    img, verite_i = simuler_photo(propre, centres, angle=float(np.random.default_rng(graine).uniform(-4, 4)), seed=graine)
    zones_i = detecter_grille(img)
    echecs += int(not zones_i)
    boites = [z for z in zones_i if z["classe"] == "event"]
    r_i, p_i = rappel_precision(zones_i, verite_i)
    total_att += len(verite_i)
    total_ok += round(r_i * len(verite_i))
    total_det += len(boites)
    total_juste += round(p_i * len(boites))
print(f"Cases attendues : {total_att}   retrouvées : {total_ok}   boîtes : {total_det}   photos sans grille : {echecs}")
print(f"RAPPEL GLOBAL   : {total_ok / total_att:.1%}")
print(f"PRÉCISION GLOBALE : {total_juste / max(1, total_det):.1%}")

## 8. Là où ça casse

Une méthode classique est efficace tant que ses **hypothèses** sont vraies. Testons-en deux.

In [ ]:
# 1) Un planning SANS bordures (export Excel sans quadrillage) : plus de traits, plus de grille
sans, c_sans = dessiner_planning(bordures=False)
print("Sans bordures :", len(detecter_grille(sans)), "boîtes")

# 2) Photo très penchée (25°) : nos noyaux « droits » ne voient plus les traits
penchee, c_pen = simuler_photo(*dessiner_planning(), angle=25)
zones_pen = detecter_grille(penchee)
print("Penchée de 25° :", len(zones_pen), "boîtes,",
      "rappel" , f"{rappel_precision(zones_pen, c_pen)[0]:.0%}" if zones_pen else "0%")
voir([sans, penchee], ["Sans bordures", "Penchée de 25°"], taille=7)

## 9. Bilan : sans IA ou avec IA ?

| | OpenCV (ce notebook) | YOLO (votre modèle) |
|---|---|---|
| Données nécessaires | aucune | quelques dizaines d'images annotées (CVAT) |
| Entraînement | aucun | oui (minutes à heures, GPU utile) |
| Rappel sur un planning net, avec bordures | très élevé, **reproductible** | dépend de vos annotations |
| Planning sans bordures, cases colorées, texte libre | échoue (il n'y a pas de « grille » à voir) | fonctionne si l'entraînement l'a vu |
| Papier plié, très penché, très courbé | dégrade vite | plus tolérant, si l'entraînement l'a vu |
| Autres mises en page (affiche, calendrier mensuel) | à réécrire | on ré-annote et on ré-entraîne |
| Explicable | oui : on sait **pourquoi** une case est ratée | boîte noire |
| Vitesse | quelques dizaines de ms, sans GPU | dépend du matériel |

**À retenir**

- L'IA n'est pas toujours nécessaire : sur un cas **cadré** (grille imprimée avec bordures), une page de morphologie
  fait aussi bien, ou mieux, qu'un modèle entraîné.
- Elle devient indispensable quand les hypothèses tombent (pas de bordures, mise en page libre, photos difficiles).
- Les deux approches se **combinent** : un détecteur classique en premier, le modèle en repli ; ou le détecteur
  classique pour pré-annoter des images dans CVAT.
- Dans les deux cas, on **mesure** (rappel, précision) sur des données que l'on connaît. Une démo qui marche une fois n'est pas une mesure.

### Pour aller plus loin

1. **Vos photos** : mettez le chemin de l'une de vos photos dans `IMAGE` (cellule 2) et refaites tourner les étapes : à quelle étape ça casse ?
2. **Réglages** : faites varier `seuil_encre`, `inclinaison`, la longueur du noyau. Que gagne-t-on en rappel ? Que perd-on en précision ?
3. **Comparer à votre modèle** : sur les mêmes photos, comptez les cases `event` retrouvées par votre champion YOLO
   (voir `test_ocr_champion.ipynb`) et par ce détecteur.
4. **Brancher la suite** : ajoutez le texte de chaque case avec l'OCR du jour 2 : les zones ont déjà le bon format
   (`classe`, `x1`, `y1`, `x2`, `y2`) pour alimenter votre parsing du jour 3.